# Universal Model Loader - One-Click Setup for Any Hugging Face Model

**Load any model with optimal settings for Google Colab**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Features

- \ud83d\udd0d Auto-detect optimal settings based on available GPU memory
- \ud83d\udfe1 Colab free tier optimized (12-16GB RAM)
- \ud83d\udd12 Supports 4-bit, 8-bit, and full precision loading
- \ud83d\udc81\u200d\ud83d\udcbb Works with any Hugging Face model
- \u2705 100% privacy - no data leaves Colab VM

---


## Configuration - Enter Your Model

**Replace the model ID below with any Hugging Face model**

Examples:
- `meta-llama/Llama-2-7b-chat-hf`
- `mistralai/Mistral-7B-Instruct-v0.2`
- `google/gemma-2-2b-it`
- `microsoft/Phi-3-mini-128k-instruct`
- `Qwen/Qwen2-7B-Instruct`
- `facebook/opt-1.3b`
- `bigcode/tiny_starcoder_py`

**Browse models**: [huggingface.co/models](https://huggingface.co/models)

---


In [ ]:
# \ud83d\udc47\ud83d\udc47\ud83d\udc47 CONFIGURE YOUR MODEL HERE \ud83d\udc47\ud83d\udc47\ud83d\udc47

# Model to load (change this to any Hugging Face model)
MODEL_ID = "gpt2"  # @param {type:"string"} description:"Hugging Face model ID"

# Authentication (required for gated models)
USE_AUTH = False  # @param {type:"boolean"} description:"Require Hugging Face login"
HF_TOKEN = ""  # @param {type:"string"} description:"Your Hugging Face token"

# Quantization settings
USE_QUANTIZATION = True  # @param {type:"boolean"} description:"Use quantization to save memory"
QUANT_BITS = 4  # @param [4, 8] description:"Quantization bits (4 = smaller, 8 = faster)"

# Device settings
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"] description:"Device to load model on"
TORCH_DTYPE = "float16"  # @param ["float32", "float16", "bfloat16"] description:"Precision type"

print(f"\ud83d\udc49 Model: {MODEL_ID}")
print(f"\ud83d\udc49 Quantization: {QUANT_BITS}-bit" if USE_QUANTIZATION else "\ud83d\udc49 Quantization: None")
print(f"\ud83d\udc49 Device: {DEVICE}")
print(f"\ud83d\udc49 Torch dtype: {TORCH_DTYPE}")

## Step 1: Security Environment Setup

In [ ]:
import os
import sys

def setup_secure_environment():
    """
    Configure secure environment for Hugging Face models.
    This runs automatically - no configuration needed.
    """
    # Disable telemetry
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    
    # Memory optimization
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
    
    # Security settings
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["HF_HOME"] = "/content/.hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/content/.hf_cache"
    
    # Disable Python bytecode caching
    sys.dont_write_bytecode = True
    
    # Create cache directory
    os.makedirs("/content/.hf_cache", exist_ok=True)
    
    print("\u2705 Security environment configured")
    print("   - Telemetry disabled")
    print("   - Secure cache enabled")
    print("   - Memory optimized")

setup_secure_environment()

## Step 2: Install Dependencies

In [ ]:
def install_dependencies():
    """Install required packages."""
    print("Installing dependencies...")
    
    # Core packages
    !pip install -q --upgrade pip
    !pip install -q transformers>=4.38.0 accelerate
    
    # Quantization support
    if USE_QUANTIZATION:
        !pip install -q bitsandbytes
    
    # Fast downloads
    !pip install -q hf_transfer
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    
    print("\u2705 Dependencies installed!")

install_dependencies()

## Step 3: Authenticate (If Required)

In [ ]:
def authenticate_hf():
    """Login to Hugging Face if required."""
    if USE_AUTH and HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("\u2705 Logged into Hugging Face!")
    else:
        print("\ud83d\udd0d No authentication (public model)")

authenticate_hf()

## Step 4: Check GPU Availability

In [ ]:
import torch

def check_gpu():
    """Display GPU information and recommendations."""
    print("=" * 50)
    print("GPU Status")
    print("=" * 50)
    
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\u2705 GPU: {gpu_name}")
        print(f"   Memory: {gpu_memory:.2f} GB")
        
        # Recommendations based on memory
        if gpu_memory < 8:
            print("\n\ud83d\udd34 Recommendations:")
            print("   - Use 4-bit quantization")
            print("   - Stick to models < 3B parameters")
            print("   - Reduce batch size")
        elif gpu_memory < 16:
            print("\n\ud83d\udfe2 Recommendations:")
            print("   - Use 4-8 bit quantization")
            print("   - Models up to 7B parameters OK")
        else:
            print("\n\u2705 You have plenty of memory!")
            print("   - Can use 8-bit or no quantization")
            print("   - Larger models supported")
    else:
        print("\u274c No GPU - Using CPU")
        print("   - Quantization highly recommended")
        print("   - Stick to very small models")
        print("   - Expect slow inference")

check_gpu()

## Step 5: Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model(model_id):
    """
    Load any Hugging Face model with optimal settings.
    """
    print(f"\n\ud83d\ude80 Loading: {model_id}")
    print("-" * 40)
    
    # Configure dtype
    dtype_map = {
        "float32": torch.float32,
        "float16": torch.float16,
        "bfloat16": torch.bfloat16
    }
    torch_dtype = dtype_map.get(TORCH_DTYPE, torch.float16)
    
    # Load tokenizer
    print("[1/2] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    print("\u2705 Tokenizer loaded")
    
    # Configure model loading
    model_kwargs = {
        "device_map": DEVICE,
        "torch_dtype": torch_dtype
    }
    
    # Add quantization if enabled
    if USE_QUANTIZATION and torch.cuda.is_available():
        print(f"\n[2/2] Loading with {QUANT_BITS}-bit quantization...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=QUANT_BITS == 4,
            load_in_8bit=QUANT_BITS == 8,
            bnb_4bit_compute_dtype=torch_dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model_kwargs["quantization_config"] = bnb_config
    else:
        print("\n[2/2] Loading without quantization...")
    
    # Load model
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
        model.eval()
        
        print("\n\u2705\u2705 Model loaded successfully!\u2705\u2705")
        print(f"\n   Parameters: {model.num_parameters():,}")
        print(f"   Device: {next(model.parameters()).device}")
        
        return model, tokenizer
        
    except Exception as e:
        print(f"\u274c Error loading model: {e}")
        print("\nTroubleshooting tips:")
        print("   1. Check if the model ID is correct")
        print("   2. Ensure you have access (for gated models)")
        print("   3. Try reducing quantization or using CPU")
        return None, None

# Load the model
model, tokenizer = load_model(MODEL_ID)

## Step 6: Generate Text

In [ ]:
def generate(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """
    Generate text with the loaded model.
    
    Args:
        prompt: Input text
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (0.1-1.0)
        top_p: Nucleus sampling threshold
    """
    if model is None or tokenizer is None:
        print("\u274c Model not loaded!")
        return
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Cleanup
    del inputs, outputs
    
    return response

print("\u2705 Generation function ready!")
print("\ud83d\udc47 Use generate('your prompt') to create text\n")

# Quick test
print("Quick test:")
result = generate("Once upon a time", max_new_tokens=50)
print(f"\ud83d\udcac {result}")

## Step 7: Memory Cleanup

In [ ]:
import gc

def cleanup():
    """Securely clear memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\u2705 Memory cleared!")

# Run cleanup
cleanup()

---

## Usage Guide

### Customizing the Model

```python
# Change at the top of Step 1
MODEL_ID = "your-model-id-here"
```

### Adjusting Generation

```python
# More focused/accurate responses
result = generate(prompt, temperature=0.3, top_p=0.8)

# More creative/varied responses
result = generate(prompt, temperature=0.9, top_p=0.95)

# Longer responses
result = generate(prompt, max_new_tokens=512)
```

### Loading Different Model Types

```python
# For text-to-image models, use:
from transformers import pipeline
pipe = pipeline("text-to-image", model="stabilityai/stable-diffusion-xl-base-1.0")

# For vision models, use:
from transformers import AutoImageProcessor, AutoModelForImageClassification
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = AutoModelForImageClassification.from_pretrained("microsoft/resnet-50")
```

---

**Troubleshooting**:

| Issue | Solution |
|-------|----------|
| OOM Error | Enable 4-bit quantization |
| Slow download | Install hf_transfer |
| Auth error | Check HF_TOKEN and model access |
| CPU only | Reduce model size, expect slower inference |

---

**Found this helpful?** \u2b50 Star [GitHub](https://github.com/unn-Known1/huggingface-colab-secure-setup)